In [32]:
import os
import json
import requests
from tqdm import tqdm
from glob import glob

# === CONFIG ===
SPECIES = [9606, 10090, 4932]  # human, mice, yeast
DATA_DIR = '../static/data'
FILE_PATTERN = os.path.join(DATA_DIR, "graph_master_part100_*.json")
files = sorted(glob(FILE_PATTERN))
print(f"Found {len(files)} files.")

def fetch_confidence_score(source, target):
    identifiers = f"{source}%0d{target}"  # URL-encoded list of IDs separated by carriage return
    results = {}

    for species in SPECIES:
        url = f"https://string-db.org/api/json/network?identifiers={identifiers}&species={species}"
        try:
            response = requests.get(url)
            if response.status_code == 404:
                # Silently skip 404 Not Found (no data for species)
                results[species] = {k: 0.0 for k in ["score", "nscore", "fscore", "pscore", "ascore", "escore", "dscore", "tscore"]}
                continue

            response.raise_for_status()
            data = response.json()

            # Find the edge between source and target
            found = False
            for edge in data:
                if (edge.get('preferredName_A') == source and edge.get('preferredName_B') == target) or \
                   (edge.get('preferredName_A') == target and edge.get('preferredName_B') == source):
                    results[species] = {
                        "score": edge.get('score', 0.0), # overall
                        "nscore": edge.get('nscore', 0.0), # neighborhood
                        "fscore": edge.get('fscore', 0.0), # fusion
                        "pscore": edge.get('pscore', 0.0), # co-occurence (phylogenetic)
                        "ascore": edge.get('ascore', 0.0), # co-expression
                        "escore": edge.get('escore', 0.0), # experimental
                        "dscore": edge.get('dscore', 0.0), # database
                        "tscore": edge.get('tscore', 0.0) # textmining
                    }
                    found = True
                    break

            if not found:
                results[species] = {k: 0.0 for k in ["score", "nscore", "fscore", "pscore", "ascore", "escore", "dscore", "tscore"]}

        except requests.exceptions.HTTPError as e:
            if response.status_code != 404:
                print(f"STRING API error for species {species}: {e}")
            results[species] = {k: 0.0 for k in ["score", "nscore", "fscore", "pscore", "ascore", "escore", "dscore", "tscore"]}

        except Exception as e:
            print(f"Unexpected error for species {species}: {e}")
            results[species] = {k: 0.0 for k in ["score", "nscore", "fscore", "pscore", "ascore", "escore", "dscore", "tscore"]}

    return results


Found 194 files.


In [33]:
test_source = "POFUT1"
test_target = "F7"

score = fetch_confidence_score(test_source, test_target)
print(f"Confidence scores for interaction {test_source} -> {test_target}: {score}")

Confidence scores for interaction POFUT1 -> F7: {9606: {'score': 0.824, 'nscore': 0, 'fscore': 0, 'pscore': 0, 'ascore': 0.047, 'escore': 0.797, 'dscore': 0, 'tscore': 0.167}, 10090: {'score': 0.0, 'nscore': 0.0, 'fscore': 0.0, 'pscore': 0.0, 'ascore': 0.0, 'escore': 0.0, 'dscore': 0.0, 'tscore': 0.0}, 4932: {'score': 0.0, 'nscore': 0.0, 'fscore': 0.0, 'pscore': 0.0, 'ascore': 0.0, 'escore': 0.0, 'dscore': 0.0, 'tscore': 0.0}}


In [34]:
import re

def extract_number(filename):
    match = re.search(r'(\d+)\.json$', filename)
    return int(match.group(1)) if match else -1

files = sorted(files, key=extract_number)

In [35]:
files[:5]

['../static/data/graph_master_part100_0.json',
 '../static/data/graph_master_part100_1.json',
 '../static/data/graph_master_part100_2.json',
 '../static/data/graph_master_part100_3.json',
 '../static/data/graph_master_part100_4.json']

In [ ]:
filelist = files[:5] # testing with 5 graph files
for file in tqdm(filelist):
    with open(file, "r") as f:
        data = json.load(f)

    links = data.get("links", [])

    for link in links:
        src = str(link["source"])
        tgt = str(link["target"])

        try: 
            scores = fetch_confidence_score(src, tgt)
            link.update(scores)
        except Exception as e:
            print(f"Error fetching score for {src} -> {tgt}: {e}")
            link.update({
                "organism": SPECIES,
                "score": 0.0,
                "nscore": 0.0,
                "fscore": 0.0,
                "pscore": 0.0,
                "ascore": 0.0,
                "escore": 0.0,
                "dscore": 0.0,
                "tscore": 0.0
            })

    # saving to new folder
    OUTPUT_DIR = "../static/data/graph_master_scored"
    filename = os.path.basename(file)
    outname = os.path.join(OUTPUT_DIR, filename)
    with open(outname, "w") as f:
        json.dump(data, f, indent=2)


 20%|██        | 1/5 [00:00<00:01,  3.82it/s]

STRING API error: 404 Client Error: Not Found for url: https://string-db.org/api/json/network?identifiers=zzzzccbbaaMBqqqqqqweweew333%0DMwewB33333&species=9606


100%|██████████| 5/5 [05:47<00:00, 69.42s/it]


In [5]:
data

{'nodes': [{'id': 'ENSG00000280178',
   'neighbors': [],
   'in_degree': [],
   'out_degree': [],
   'links': [],
   'node_neighbor_count': 0},
  {'id': 'ENSG00000280267',
   'neighbors': [],
   'in_degree': [],
   'out_degree': [],
   'links': [],
   'node_neighbor_count': 0},
  {'id': 'ENSG00000280297',
   'neighbors': [],
   'in_degree': [],
   'out_degree': [],
   'links': [],
   'node_neighbor_count': 0},
  {'id': 'ENSG00000280433',
   'neighbors': [],
   'in_degree': [],
   'out_degree': [],
   'links': [],
   'node_neighbor_count': 0},
  {'id': 'ENTHD1',
   'neighbors': ['ENTHD2', 'SUPT7L', 'RASD2', 'RAPGEF2', 'SUPT6H'],
   'in_degree': ['ENTHD2'],
   'out_degree': ['ENTHD2', 'RAPGEF2', 'RASD2', 'SUPT6H', 'SUPT7L'],
   'links': [{'source': {'id': 'ENTHD2'},
     'target': {'id': 'ENTHD1'},
     'source_neighbors': 0,
     'target_neighbors': 1},
    {'source': {'id': 'ENTHD1'},
     'target': {'id': 'ENTHD2'},
     'source_neighbors': 1,
     'target_neighbors': 0},
    {'source